# 📘 Pré-entraînement YOLO compost — sur les datasets EXTERNES (centre de tri)

Entraîne un modèle sur les datasets web (zerowaste, taco, proj3, warp) et produit un
`best.pt` **pré-entraîné** sauvegardé sur Drive. Le **fine-tuning sur nos captures** se fait
dans le notebook séparé `colab_finetune.ipynb`.

Prérequis : runtime GPU ; secret `GITHUB_TOKEN` ; zips `MyDrive/compost/dataset_raw_<nom>.zip`.

In [ ]:
# 1. Clone du repo (token lu depuis les Secrets Colab — utilisé uniquement pour cloner)
BRANCH = 'yolo'   # branche de travail ; mettre 'main' après fusion
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
%cd /content
!rm -rf /content/repo
!git clone --depth 1 --branch {BRANCH} https://{token}@github.com/TSResearch-hub/Compost_Waste_Yolo.git /content/repo
%cd /content/repo/compost-yolo

In [ ]:
# 2. Installation des dépendances
!pip install -q -e .

In [ ]:
# 3. Données : datasets EXTERNES -> /content/dataset
TRAIN_DATASETS = ['zerowaste', 'taco', 'proj3', 'warp']

import os, shutil
from google.colab import drive
# Montage robuste (vide un /content/drive résiduel non monté avant de remonter)
if os.path.isdir('/content/drive') and not os.path.ismount('/content/drive'):
    shutil.rmtree('/content/drive', ignore_errors=True)
drive.mount('/content/drive')

for name in TRAIN_DATASETS:
    zip_path = f'/content/drive/MyDrive/compost/dataset_raw_{name}.zip'
    if not os.path.exists(zip_path):
        print(f'⚠️  {name} : {zip_path} INTROUVABLE sur Drive — ignoré (à uploader)')
        continue
    !cp {zip_path} /content/
    !unzip -q -o /content/dataset_raw_{name}.zip -d /content/dataset_raw_{name}
    !python scripts/prepare_dataset.py --source /content/dataset_raw_{name} --output /content/dataset

In [ ]:
# 3b. Histogramme par dataset externe INDIVIDUEL — instances par classe
import yaml
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt

names = yaml.safe_load(open('configs/data.yaml'))['names']
x = range(len(names))
for name in TRAIN_DATASETS:
    counts = Counter()
    for lf in Path(f'/content/dataset_raw_{name}/labels').glob('*.txt'):
        for line in lf.read_text().splitlines():
            if line.strip():
                counts[int(line.split()[0])] += 1
    if not counts:
        print(f"⚠️  {name} : aucun label (zip manquant ?) — sauté"); continue
    fig, ax = plt.subplots(figsize=(10, 3))
    b = ax.bar(list(x), [counts[j] for j in x]); ax.bar_label(b, fmt='%d', padding=2)
    ax.set_yscale('log'); ax.set_xticks(list(x)); ax.set_xticklabels(names, rotation=20)
    ax.set_ylabel('instances (log)'); ax.set_title(f"Dataset externe « {name} »")
    ax.grid(axis='y', alpha=0.3); fig.tight_layout(); plt.show()
    print(f"{name}: {sum(counts.values())} —", ", ".join(f"{names[j]}:{counts[j]}" for j in x))

In [ ]:
# 3c. EXTERNES (tous) — instances par classe + nombre de boîtes par image
import yaml
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt

names = yaml.safe_load(open('configs/data.yaml'))['names']; x = range(len(names))
pc, bpi, n_img, n_neg = Counter(), [], 0, 0
for name in TRAIN_DATASETS:
    idir, ldir = Path(f'/content/dataset_raw_{name}/images'), Path(f'/content/dataset_raw_{name}/labels')
    for img in idir.glob('*'):
        if img.suffix.lower() not in ('.jpg','.jpeg','.png'): continue
        n_img += 1
        lf = ldir / f'{img.stem}.txt'
        lines = [l for l in lf.read_text().splitlines() if l.strip()] if lf.exists() else []
        bpi.append(len(lines)); n_neg += (not lines)
        for l in lines: pc[int(l.split()[0])] += 1
fig, axes = plt.subplots(1, 2, figsize=(14, 3.6))
b = axes[0].bar(list(x), [pc[j] for j in x]); axes[0].bar_label(b, fmt='%d', padding=2)
if any(pc.values()): axes[0].set_yscale('log')
axes[0].set_xticks(list(x)); axes[0].set_xticklabels(names, rotation=20)
axes[0].set_title('Externes — instances par classe'); axes[0].grid(axis='y', alpha=0.3)
cap=10; dist=Counter(min(k,cap) for k in bpi); xs=list(range(cap+1))
b2=axes[1].bar(xs,[dist[k] for k in xs]); axes[1].bar_label(b2, fmt='%d', padding=2)
axes[1].set_xticks(xs); axes[1].set_xticklabels([str(k) for k in xs[:-1]]+[f'{cap}+'])
axes[1].set_xlabel('boîtes/image'); axes[1].set_title('Externes — boîtes par image'); axes[1].grid(axis='y', alpha=0.3)
fig.tight_layout(); plt.show()
moy = sum(bpi)/len(bpi) if bpi else 0
print(f"Externes : {n_img} images dont {n_neg} négatives | {sum(pc.values())} instances | {moy:.2f} boîtes/image")
print("  ", ", ".join(f"{names[j]}:{pc[j]}" for j in x))

In [ ]:
# 3d. Composition d'ENTRAÎNEMENT (/content/dataset) — par classe et par split
import yaml
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt

names = yaml.safe_load(open('configs/data.yaml'))['names']; splits=['train','val','test']
counts = {s: Counter() for s in splits}
for s in splits:
    for lf in Path(f'/content/dataset/labels/{s}').glob('*.txt'):
        for line in lf.read_text().splitlines():
            if line.strip(): counts[s][int(line.split()[0])] += 1
x=range(len(names)); w=0.27
fig, ax = plt.subplots(figsize=(12,4))
for i,s in enumerate(splits):
    b=ax.bar([v+(i-1)*w for v in x],[counts[s][j] for j in x],w,label=s); ax.bar_label(b,fmt='%d',padding=2,fontsize=7)
if any(any(c.values()) for c in counts.values()): ax.set_yscale('log')
ax.set_xticks(list(x)); ax.set_xticklabels(names, rotation=20); ax.legend(); ax.grid(axis='y', alpha=0.3)
ax.set_title('Entraînement (externes) — par classe et par split'); fig.tight_layout(); plt.show()
for s in splits:
    print(f"{s}: {sum(counts[s].values())} —", ", ".join(f"{names[j]}:{counts[s][j]}" for j in x))

In [ ]:
# 4. Pré-entraînement (100 epochs ; checkpoints sur Drive toutes les 10 epochs)
# MODEL : 'yolov8n.pt' (défaut) ou 'rtdetr-l.pt' (RT-DETR — réduire --batch si mémoire)
# Reprise après coupure : ajouter --resume /content/runs/pretrain_xxx/weights/last.pt
MODEL = 'yolov8n.pt'
!python scripts/train.py --model {MODEL} --data /content/dataset/data.yaml \
    --run-prefix pretrain --runs-dir /content/runs \
    --backup-dir /content/drive/MyDrive/compost/backups --backup-every 10

In [ ]:
# 5. ÉVAL — test des datasets EXTERNES (perf générale, PAS le compost réel)
from pathlib import Path
cands = []
for d in ['/content/runs','/content/drive/MyDrive/compost/runs','/content/drive/MyDrive/compost/backups']:
    cands += Path(d).glob('pretrain_*/weights/best.pt')
    cands += Path(d).glob('train_*/weights/best.pt')   # anciens runs, avant le renommage
assert cands, "Aucun best.pt trouvé — l'entraînement a-t-il tourné ?"
best = sorted(cands, key=lambda p: p.stat().st_mtime)[-1]
print('Modèle évalué (externes) :', best)
!python scripts/evaluate.py --weights {best} --data /content/dataset/data.yaml --split test --runs-dir /content/runs
from IPython.display import Image, display
evals = sorted(Path('/content/runs').glob('eval_*test_*'), key=lambda p: p.stat().st_mtime)
if evals:
    for img in sorted(evals[-1].rglob('*confusion*.png')): print(img.name); display(Image(str(img)))

In [ ]:
# 6. Copie du run complet vers Drive (le best.pt PRÉ-ENTRAÎNÉ servira au fine-tuning)
!mkdir -p /content/drive/MyDrive/compost/runs
!cp -r /content/runs/* /content/drive/MyDrive/compost/runs/
!ls /content/drive/MyDrive/compost/runs
print("\\n>>> Télécharge le best.pt pré-entraîné vers models/pretrain_<model>.pt")
print(">>> sur ta machine (ex. models/pretrain_yolov8n.pt) : c'est le point de")
print(">>> départ canonique de scripts/retrain.py <<<")